# v21 案B: 遷移(dip事前)を学習

PFの遷移 `rate_n = a*rate + noise` に、**学習ドリフト** `drift[md_i]`(期待dip=dgeo_z/dMD)を混合:
`rate_n = (1-lam)*(a*rate) + lam*drift + noise`。
- ドリフトNN(MLP)入力 = 推論可能な特徴のみ: md_since(ラテラル進行度) / 既知dip(ir) / dz/dMD / Z / GR / GR傾き。
- 学習: trainの真 geo_z=TVT+Z の局所dip を回帰(PF微分不要、NNはPFの外で1回/坑井)。
- 狙う失敗: **過外挿**(「ラテラルで水平化(dip→0, geo_z は Z に追従)」を学習し行き過ぎを抑える)。
- 評価: 素のr0 PF と overall / 硬い核 per-well を lam を振って比較(ノイズ床±0.2、桁変化を見る)。

In [ ]:
import sys, time, numpy as np, torch, torch.nn as nn
sys.path.insert(0, "v21")
import importlib, v21_common as C; importlib.reload(C)
torch.manual_seed(0); np.random.seed(0)
DEV = C.DEVICE; print("device:", DEV)

N_TRAIN_WELLS = 300   # ドリフト学習に使う坑井数
ns = C.load_v9_ns()
p  = C.r0_params(ns, use_phys=False, n_seeds=8)
WELLS = C.list_train_wells()

In [ ]:
# ===== 特徴量(推論可能列のみ)とターゲット(真dip)を作る =====
def feats_of(inp):
    """eval各行の特徴 (T,F)。md_since/ir/Z/dz_dmd/GR/GR傾き。"""
    md = inp["md"].astype(np.float64); z = inp["z"].astype(np.float64); gr = inp["gr"].astype(np.float64)
    mds = inp["md_since"].astype(np.float64); ir = float(inp["ir"])
    dz = np.gradient(z, md); dgr = np.gradient(gr, md)
    F = np.stack([
        mds/1000.0,                       # ラテラル進行度(kft)
        np.full_like(mds, ir),            # 既知dip(well定数)
        (z - z[0])/1000.0,                # 相対Z(kft)
        dz,                               # dZ/dMD
        (gr - np.nanmean(gr))/(np.nanstd(gr)+1e-6),
        dgr,                              # GR傾き
        1.0 - np.exp(-mds/300.0),         # 水平化の進行(0→1)
    ], axis=1).astype(np.float32)
    return F

def target_dip(inp):
    """真 geo_z=TVT+Z の局所dip dgeo_z/dMD。"""
    if inp["tvt_true"] is None: return None
    md = inp["md"].astype(np.float64); geo = inp["tvt_true"].astype(np.float64) + inp["z"].astype(np.float64)
    return np.gradient(geo, md).astype(np.float32)

# 学習データ蓄積
Xs=[]; Ys=[]; t0=time.time()
for i,wid in enumerate(WELLS[:N_TRAIN_WELLS]):
    inp = C.build_inputs(wid, ns, p)
    if inp is None or inp["tvt_true"] is None: continue
    y = target_dip(inp)
    if y is None: continue
    X = feats_of(inp)
    m = np.isfinite(X).all(1) & np.isfinite(y)
    Xs.append(X[m]); Ys.append(y[m])
    if (i+1)%50==0: print(f"  {i+1}/{N_TRAIN_WELLS} ({time.time()-t0:.0f}s)")
X = np.concatenate(Xs); Y = np.concatenate(Ys)
FMEAN = X.mean(0); FSTD = X.std(0)+1e-6
Xn = (X-FMEAN)/FSTD
print("学習サンプル:", X.shape, "| dip中央", float(np.median(Y)), "範囲", float(Y.min()), float(Y.max()))

In [ ]:
# ===== ドリフトMLP =====
NF = X.shape[1]
class Drift(nn.Module):
    def __init__(self, nf):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(nf,64), nn.ReLU(), nn.Linear(64,64), nn.ReLU(), nn.Linear(64,1))
    def forward(self, x): return self.net(x).squeeze(-1)

mlp = Drift(NF).to(DEV)
opt = torch.optim.Adam(mlp.parameters(), lr=1e-3)
Xt = torch.tensor(Xn, device=DEV); Yt = torch.tensor(Y, device=DEV)
EPOCHS=40; BS=8192
for ep in range(EPOCHS):
    perm = torch.randperm(len(Xt), device=DEV); tot=0; nb=0
    for s in range(0,len(perm),BS):
        idx=perm[s:s+BS]; pred=mlp(Xt[idx]); loss=nn.functional.smooth_l1_loss(pred, Yt[idx])
        opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item(); nb+=1
    if ep%5==0 or ep==EPOCHS-1: print(f"ep{ep} huber={tot/nb:.5f}")
torch.save({"mlp":mlp.state_dict(),"FMEAN":FMEAN,"FSTD":FSTD}, "v21/drift_B.pt")
print("saved v21/drift_B.pt")

In [ ]:
# ===== drift_fn: 学習MLPで eval各行の期待dip drift[T] を生成 =====
@torch.no_grad()
def drift_fn(wid, inp):
    mlp.eval()
    X = feats_of(inp); X = np.nan_to_num((X-FMEAN)/FSTD, nan=0.0).astype(np.float32)
    d = mlp(torch.tensor(X, device=DEV)).float().cpu().numpy()
    return d
print("drift_fn ready")

In [ ]:
# ===== 評価: 素のr0 PF vs 案B(学習遷移, lam スイープ) =====
EVAL = WELLS[:40] + C.HARD_WELLS
EVAL = list(dict.fromkeys(EVAL))
base = C.eval_wells(EVAL, ns, p, chunk=4)
results = {0.0: base}
for lam in [0.3, 0.6, 0.9]:
    results[lam] = C.eval_wells(EVAL, ns, p, chunk=4, drift_fn=drift_fn, lam_drift=lam)

print("\n=== overall(lam スイープ) ===")
for lam,r in results.items():
    print(f"  lam={lam:.1f}: overall={r['overall']:.3f}  d={r['overall']-base['overall']:+.3f}")

bestlam = min([0.3,0.6,0.9], key=lambda L: results[L]["overall"])
B = results[bestlam]
print(f"\n=== 硬い核 per-well (base vs B lam={bestlam}) ===")
print(f"{'well':10s} {'base':>7s} {'B':>7s} {'d':>7s}")
for w in C.HARD_WELLS:
    if w in base["perwell"] and w in B["perwell"]:
        b=base["perwell"][w]; a=B["perwell"][w]
        print(f"{w:10s} {b:7.2f} {a:7.2f} {a-b:+7.2f}")
hb=[base["perwell"][w] for w in C.HARD_WELLS if w in base["perwell"]]
ha=[B["perwell"][w]    for w in C.HARD_WELLS if w in B["perwell"]]
print(f"\nhard mean: base={np.mean(hb):.2f}  B(lam={bestlam})={np.mean(ha):.2f}")